# EDA on Online Retail Sales

**OASIS INFOBYTE SIP — Data Analytics Level 1 • Task 1**

This notebook performs data inspection, cleaning, descriptive statistics, monthly and quarterly sales trends, product analysis, country analysis, correlation analysis and business recommendations.

### Important dataset limitations
- The supplied dataset has **no age or gender fields**, so age/gender analysis cannot be performed honestly.
- The supplied dataset has **no dedicated product-category field**, so product-category revenue is not invented or inferred.
- **Best-selling product = highest total Quantity Sold.** Revenue ranking is a separate analysis.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
BASE = Path.cwd()
RAW_DATA = BASE / 'data' / 'raw' / 'online_retail.csv'
CLEANED_DATA = BASE / 'data' / 'cleaned' / 'online_retail_cleaned.csv'
OUT = BASE / 'outputs'; OUT.mkdir(exist_ok=True)
CLEANED_DATA.parent.mkdir(exist_ok=True)
df_raw = pd.read_csv(RAW_DATA, encoding='latin1')
df_raw.columns = df_raw.columns.str.strip()
for col in ['Description','Country','StockCode','InvoiceNo']:
    df_raw[col] = df_raw[col].astype('string').str.strip()
df_raw['InvoiceDate'] = pd.to_datetime(df_raw['InvoiceDate'], errors='coerce')
print(f'Raw shape: {df_raw.shape}')
print('\nColumn data types:')
print(df_raw.dtypes.to_string())
print(f'\nExact duplicate rows: {df_raw.duplicated().sum()}')
print('\nMissing values:')
print(df_raw.isna().sum().sort_values(ascending=False).to_string())


Raw shape: (541909, 9)

Column data types:
index                   int64
InvoiceNo      string[python]
StockCode      string[python]
Description    string[python]
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country        string[python]
Exact duplicate rows: 0

Missing values:
CustomerID     135080
Description      1454
index               0
StockCode           0
InvoiceNo           0
Quantity            0
InvoiceDate         0
UnitPrice            0
Country              0


## 1. Data cleaning


In [2]:
df = df_raw.drop_duplicates().copy()
df['is_cancelled'] = df['InvoiceNo'].astype('string').str.upper().str.startswith('C', na=False)
df['Revenue'] = df['Quantity'] * df['UnitPrice']
sales_df = df[(~df['is_cancelled']) & (df['Quantity'] > 0) & (df['UnitPrice'] > 0) & df['InvoiceDate'].notna()].copy()
sales_df.to_csv(CLEANED_DATA, index=False)
print(f'Clean sales rows: {len(sales_df):,}')
print(f'Total revenue: £{sales_df["Revenue"].sum():,.2f}')
print(f'Unique invoices: {sales_df["InvoiceNo"].nunique():,}')
print(f'Unique products (stock codes): {sales_df["StockCode"].nunique():,}')
print(f'Unique countries: {sales_df["Country"].nunique():,}')


Clean sales rows: 530,104
Total revenue: £10,666,684.54
Unique invoices: 19,960
Unique products (stock codes): 3,922
Unique countries: 38


## 2. Descriptive statistics

The OASIS requirement asks for mean, median, mode and standard deviation for numerical columns. For business analysis, the relevant numerical measures are **Quantity, UnitPrice and Revenue**. The `index` column is a technical row identifier from the CSV, not a business variable, so it is intentionally excluded from the descriptive statistics.


In [3]:
numeric_cols = ['Quantity','UnitPrice','Revenue']
stats = pd.DataFrame({'Mean': sales_df[numeric_cols].mean(), 'Median': sales_df[numeric_cols].median(), 'Mode': sales_df[numeric_cols].mode().iloc[0], 'Standard Deviation': sales_df[numeric_cols].std()}).round(2)
print(stats.to_string())


            Mean  Median   Mode  Standard Deviation
Quantity   10.54    3.00   1.00              155.52
UnitPrice   3.91    2.08   1.25               35.92
Revenue    20.12    9.90  15.00              270.36


## 3. Monthly and quarterly sales trends


In [4]:
sales_df['Month'] = sales_df['InvoiceDate'].dt.to_period('M').astype(str)
sales_df['Quarter'] = sales_df['InvoiceDate'].dt.to_period('Q').astype(str)
monthly = sales_df.groupby('Month')['Revenue'].sum()
quarterly = sales_df.groupby('Quarter')['Revenue'].sum()
fig, ax = plt.subplots(figsize=(12,5)); monthly.plot(marker='o', ax=ax); ax.set_title('Monthly Revenue Trend'); ax.set_xlabel('Month'); ax.set_ylabel('Revenue (£)'); plt.xticks(rotation=45); plt.tight_layout(); plt.savefig(OUT/'monthly_revenue_trend.png', dpi=160, bbox_inches='tight'); plt.show()
fig, ax = plt.subplots(figsize=(11,5)); quarterly.plot(marker='o', ax=ax); ax.set_title('Quarterly Revenue Trend'); ax.set_xlabel('Quarter'); ax.set_ylabel('Revenue (£)'); plt.xticks(rotation=45); plt.tight_layout(); plt.savefig(OUT/'quarterly_revenue_trend.png', dpi=160, bbox_inches='tight'); plt.show()
print(f'Peak month: {monthly.idxmax()} — £{monthly.max():,.2f}')
print(f'Peak quarter: {quarterly.idxmax()} — £{quarterly.max():,.2f}')


Peak month: 2011-11 — £1,509,496.33
Peak quarter: 2011Q4 — £3,303,268.31


### Monthly Revenue Trend — Interpretation

Revenue reaches its highest monthly level in **2011-11**. The strong late-year performance suggests a pronounced seasonal demand pattern, so inventory, staffing and promotional planning should account for the year-end peak.

![Monthly Revenue Trend](outputs/monthly_revenue_trend.png)

### Quarterly Revenue Trend — Interpretation

**2011Q4** is the strongest quarter, confirming that the late-year surge is not limited to a single month. This supports planning additional capacity and stock availability ahead of Q4.


## 4. Country analysis — additional insight


In [5]:
country_revenue = sales_df.groupby('Country')['Revenue'].sum().sort_values(ascending=False)
country_orders = sales_df.groupby('Country')['InvoiceNo'].nunique().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10,6)); country_revenue.head(10).sort_values().plot(kind='barh', ax=ax); ax.set_title('Top 10 Countries by Revenue'); ax.set_xlabel('Revenue (£)'); plt.tight_layout(); plt.savefig(OUT/'top_10_countries_by_revenue.png', dpi=160, bbox_inches='tight'); plt.show()
fig, ax = plt.subplots(figsize=(10,6)); country_orders.head(10).sort_values().plot(kind='barh', ax=ax); ax.set_title('Top 10 Countries by Number of Orders'); ax.set_xlabel('Unique invoices'); plt.tight_layout(); plt.savefig(OUT/'top_10_countries_by_orders.png', dpi=160, bbox_inches='tight'); plt.show()
print(f'United Kingdom revenue share: {country_revenue["United Kingdom"]/country_revenue.sum():.1%}')
print(f'Top-10-country revenue share: {country_revenue.head(10).sum()/country_revenue.sum():.1%}')
print(f'United Kingdom unique invoices: {country_orders["United Kingdom"]:,}')


United Kingdom revenue share: 84.6%
Top-10-country revenue share: 97.2%
United Kingdom unique invoices: 18,019


### Country Revenue — Interpretation

The **United Kingdom contributes the majority of total revenue**, while the top 10 countries contribute almost all revenue. Revenue is therefore highly concentrated in a small number of markets.

![Top 10 Countries by Revenue](outputs/top_10_countries_by_revenue.png)

### Country Orders — Interpretation

The United Kingdom also leads by order count. This reinforces the importance of the UK market while highlighting the need for selective international expansion rather than treating all markets equally.

![Top 10 Countries by Orders](outputs/top_10_countries_by_orders.png)


## 5. Product analysis


In [6]:
non_product_codes = {'POST','DOT','C2','23444','BANK CHARGES','AMAZONFEE','B','S','D','M','m'}
product_sales = sales_df[~sales_df['StockCode'].isin(non_product_codes)].copy()
top_products_units = product_sales.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)
top_products_revenue = product_sales.groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(10)
print(f'Best-selling product by Quantity Sold: {top_products_units.index[0]} — {top_products_units.iloc[0]:,} units')
print(f'Top product by revenue after non-product exclusion: {top_products_revenue.index[0]} — £{top_products_revenue.iloc[0]:,.2f}')
print('Excluded non-product/service stock codes: POST, DOT, C2, 23444, BANK CHARGES, AMAZONFEE, B, S, D, M/m')
fig, ax = plt.subplots(figsize=(10,6)); top_products_units.sort_values().plot(kind='barh', ax=ax); ax.set_title('Top 10 Products by Units Sold'); ax.set_xlabel('Units sold'); plt.tight_layout(); plt.savefig(OUT/'top_10_products_by_units.png', dpi=160, bbox_inches='tight'); plt.show()
fig, ax = plt.subplots(figsize=(10,6)); top_products_revenue.sort_values().plot(kind='barh', ax=ax); ax.set_title('Top 10 Products by Revenue (Non-Product Lines Excluded)'); ax.set_xlabel('Revenue (£)'); plt.tight_layout(); plt.savefig(OUT/'top_10_products_by_revenue.png', dpi=160, bbox_inches='tight'); plt.show()


Best-selling product by Quantity Sold: PAPER CRAFT , LITTLE BIRDIE — 80,995 units
Top product by revenue after non-product exclusion: REGENCY CAKESTAND 3 TIER — £174,484.74
Excluded non-product/service stock codes: POST, DOT, C2, 23444, BANK CHARGES, AMAZONFEE, B, S, D, M/m


### Top 10 Products by Units Sold — Interpretation

**Best-selling is defined by total Quantity Sold.** The volume leader is identified from total units sold, not monetary value.

![Top 10 Products by Units Sold](outputs/top_10_products_by_units.png)

### Top 10 Products by Revenue — Corrected Interpretation

After removing non-product lines such as **postage, carriage, fees, discounts and manual/administrative entries**, the revenue ranking reflects product sales rather than logistics or administrative charges.

The revenue leader can differ from the unit-volume leader, showing why both **Quantity Sold** and **Revenue** should be reported separately.

![Top 10 Products by Revenue](outputs/top_10_products_by_revenue.png)

### Product Category Limitation

The supplied dataset has **no dedicated product-category field**. A category revenue chart would require invented or manually inferred categories, so **product category analysis is not reported**.


## 6. Correlation analysis


In [7]:
corr = sales_df[['Quantity','UnitPrice','Revenue']].corr()
print(corr.round(2).to_string())
fig, ax = plt.subplots(figsize=(8,6)); sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax); ax.set_title('Correlation Matrix'); plt.tight_layout(); plt.savefig(OUT/'correlation_heatmap.png', dpi=160, bbox_inches='tight'); plt.show()


           Quantity  UnitPrice  Revenue
Quantity       1.00      -0.00     0.91
UnitPrice     -0.00       1.00     0.14
Revenue        0.91       0.14     1.00


### Correlation Heatmap — Interpretation

The heatmap shows the relationship between Quantity, UnitPrice and Revenue. These relationships should **not** be interpreted as independent causal effects because Revenue is mechanically calculated as **Quantity × UnitPrice**.

![Correlation Heatmap](outputs/correlation_heatmap.png)


## 7. Dataset limitations

### Age and gender
The supplied dataset contains **no age or gender variables**. Therefore, customer segmentation by age or gender cannot be performed reliably. No demographic assumptions are made.

### Product category
The supplied dataset contains **no dedicated product-category field**. Product-category revenue cannot be calculated without creating unsupported classifications, so this analysis is intentionally omitted.

### Customer identification
CustomerID is missing for a substantial number of transactions, which limits customer-level retention, frequency and lifetime-value analysis.


## 8. Findings and actionable recommendations

- **Peak period:** 2011-11 and 2011Q4 are the strongest revenue periods; plan inventory and staffing before Q4.
- **Market concentration:** the UK is the dominant market; maintain service levels there while selectively testing international growth.
- **Product volume vs value:** track units sold and revenue separately because the volume leader and revenue leader can differ.
- **Operational discipline:** exclude cancellations and non-product/service lines when measuring product performance.
- **Data quality:** capture more complete CustomerID values and add demographic/category fields in future data collection if those analyses are required.
